## **Loan Approval System with SHAP Explanations**


**Author**: Lisa van der Linden  

**Bachelor of Science** - Applied Artificial Intelligence

**Bachelor Thesis:** The Effect of Explainable Artificial Intelligence (XAI) on User Trust in Deep Neural Network-Based Loan Approval Systems

**International University of Applied Sciences**

The purpose of this notebook is generate explanations in various levels of transparancy for automated loan approval evaluations.

A DNN model is trained using the German Credit Dataset. Loan applications will be evaluated using binary outcomes (approved/rejected).

Deep SHAP will be implemented to generate explanations for the predictions. Outputs of these  explanations will be generated in three levels of incremental transparancy:
- The Amber System: No explanation
- The Opal System: a simplified explanation in plain english using the top two features based on SHAP values
- The Quartz System: The full SHAP waterfall

The code contains an algorithm to select scenarios with a variety of top features and good explainability.

The loan approval model is intended for demonstration and resarch purposes only.

# Setup and Data Loading

In [ ]:
# Install required packages
!pip install shap scikit-learn pandas numpy matplotlib seaborn tensorflow --quiet

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
import warnings
import os
import json
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

print("✓ Libraries imported successfully")

# Feature Name Mapping Dictionary

Translate technical feature codes into readable descriptions.

All currency values converted to EUR

In [ ]:
# Feature name mapping
FEATURE_NAMES = {
    'checking_status': 'Checking Account Status',
    'duration': 'Loan Duration',
    'credit_history': 'Credit History',
    'purpose': 'Loan Purpose',
    'credit_amount': 'Loan Amount',
    'savings_status': 'Savings Account Balance',
    'employment': 'Employment Duration',
    'installment_rate': 'Installment Rate',
    'personal_status': 'Personal Status',
    'other_parties': 'Other Parties',
    'residence_since': 'Residence Duration',
    'property_magnitude': 'Property Ownership',
    'age': 'Age',
    'other_payment_plans': 'Other Payment Plans',
    'housing': 'Housing Status',
    'existing_credits': 'Number of Existing Credits',
    'job': 'Job Status',
    'num_dependents': 'Number of Dependents'
}

# Value mappings for categorical features and EUR conversion
VALUE_MAPPINGS = {
    'checking_status': {
        'A11': 'less than €0 (overdrawn)',
        'A12': '€0 to €200',
        'A13': 'more than €200',
        'A14': 'no checking account'
    },
    'credit_history': {
        'A30': 'no credits taken/all credits paid back',
        'A31': 'all credits paid back',
        'A32': 'existing credits paid back',
        'A33': 'delayed payments in the past',
        'A34': 'critical account/other credits existing'
    },
    'purpose': {
        'A40': 'new car',
        'A41': 'used car',
        'A42': 'furniture/equipment',
        'A43': 'radio/television',
        'A44': 'domestic appliances',
        'A45': 'repairs',
        'A46': 'education',
        'A47': 'vacation',
        'A48': 'retraining',
        'A49': 'business',
        'A410': 'other'
    },
    'savings_status': {
        'A61': 'less than €50',
        'A62': '€50 to €250',
        'A63': '€250 to €500',
        'A64': 'more than €500',
        'A65': 'unknown/no savings account'
    },
    'employment': {
        'A71': 'unemployed',
        'A72': 'less than 1 year',
        'A73': '1 to 4 years',
        'A74': '4 to 7 years',
        'A75': 'more than 7 years'
    },
    'housing': {
        'A151': 'rent',
        'A152': 'own',
        'A153': 'free'
    },
    'job': {
        'A171': 'unemployed/unskilled non-resident',
        'A172': 'unskilled resident',
        'A173': 'skilled employee/official',
        'A174': 'management/self-employed'
    }
}

print("Feature mappings loaded")


### Load German Credit Dataset

In [ ]:
# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
column_names = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_status', 'employment', 'installment_rate', 'personal_status',
    'other_parties', 'residence_since', 'property_magnitude', 'age',
    'other_payment_plans', 'housing', 'existing_credits', 'job',
    'num_dependents', 'own_telephone', 'foreign_worker', 'class'
]

# Load data
df = pd.read_csv(url, sep=' ', names=column_names)

# Convert target: 1=good credit (approve), 2=bad credit (reject)
df['class'] = df['class'].map({1: 1, 2: 0})

print("\n   Removing irrelevant features: foreign_worker, own_telephone")
print("   - foreign_worker: Near-constant values")
print("   - own_telephone: Minimal predictive value\n")

# Remove irrelevant features
df = df.drop(['foreign_worker', 'own_telephone'], axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['class'].value_counts()}")
print(f"\nApproval rate: {df['class'].mean():.1%}")
print(f"\nFeatures included in model: {len(df.columns) - 1}")

# Store original data for later use
df_original = df.copy()
df.head()


## Data Preprocessing and Model Training

In [ ]:
# Separate features and target
X = df.drop('class', axis=1)
y = df['class']

# Encode categorical variables
label_encoders = {}
X_encoded = X.copy()

for column in X_encoded.columns:
    if X_encoded[column].dtype == 'object':
        le = LabelEncoder()
        X_encoded[column] = le.fit_transform(X_encoded[column])
        label_encoders[column] = le

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Keep original (non-scaled) versions for profile generation
X_train_original = X.loc[X_train.index]
X_test_original = X.loc[X_test.index]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames with preserved indices
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"✓ Data preprocessed")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Number of features: {X_train.shape[1]}")
print(f"✓ Indices preserved in X_test: {X_test.index[:5].tolist()}")


In [ ]:
# Build Deep Neural Network
model = keras.Sequential([
    # First hidden layer: 64 neurons with ReLU activation
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),  # Dropout for regularization

    # Second hidden layer: 32 neurons
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),

    # Third hidden layer: 16 neurons
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),

    # Output layer: sigmoid for binary classification (0-1 probability)
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model architecture defined")
print(f"Architecture: {X_train.shape[1]} -> 64 -> 32 -> 16 -> 1")
model.summary()


In [ ]:
# Training the model
# Calculate class weights to handle imbalanced dataset (70% approved, 30% rejected)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))
print(f"Class weights: {class_weights}")
print(f"  - Class 0 (Rejected): {class_weights[0]:.3f}")
print(f"  - Class 1 (Approved): {class_weights[1]:.3f}")

# Define callbacks for training optimization
callbacks = [
    # Early Stopping: Stop training when validation loss stops improving
    # patience=40
    # restore_best_weights ensures we keep the best model
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=40,
        restore_best_weights=True,
        verbose=1
    ),

    # Learning Rate Scheduler: Reduce learning rate when progress stalls
    # If val_loss doesn't improve for 10 epochs, multiply LR by 0.5
    # This allows fine-tuning when the model approaches optimal weights
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,        # Reduce LR by half
        patience=10,       # Wait 10 epochs before reducing
        min_lr=0.00001,    # Don't reduce below this value
        verbose=1
    )
]

# Train model
history = model.fit(
    X_train.values, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=callbacks,
    class_weight=class_weights
)

print("\nModel training complete")
print(f"Training stopped at epoch: {len(history.history['loss'])}")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.3f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.3f}")


# Loading Permormance Metrics

In [ ]:
# Generate predictions
y_pred_proba_array = model.predict(X_test.values, verbose=0).flatten()
y_pred_array = (y_pred_proba_array > 0.5).astype(int)

# Convert to pandas Series with preserved indices
y_pred_proba = pd.Series(y_pred_proba_array, index=X_test.index)
y_pred = pd.Series(y_pred_array, index=X_test.index)

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

print(f"\nPredicted approvals: {y_pred.sum()} ({y_pred.mean()*100:.1f}%)")
print(f"\nPredicted rejections: {(1-y_pred).sum()} ({(1-y_pred.mean())*100:.1f}%)")


In [ ]:
# Calculate all metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_pred_proba)

# Create metrics dictionary
metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall (Sensitivity)', 'F1-Score', 'AUC-ROC'],
    'Value': [accuracy, precision, recall, f1, auc_roc],
}

# Create DataFrame
metrics_df = pd.DataFrame(metrics_data)
metrics_df['Value'] = metrics_df['Value'].apply(lambda x: f"{x:.4f}")

# Display table
print("=" * 70)
print("MODEL PERFORMANCE METRICS")
print("=" * 70)
print(metrics_df.to_string(index=False))
print("=" * 70)

# Save to CSV
metrics_csv_path = "model_performance_metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"\nMetrics saved to: {metrics_csv_path}")

# Also create a simple version for easy copying
metrics_simple = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
    'Value': [f"{accuracy:.4f}", f"{precision:.4f}", f"{recall:.4f}", f"{f1:.4f}", f"{auc_roc:.4f}"]
})
metrics_simple.to_csv("model_metrics_simple.csv", index=False)

# TRAINING SUMMARY

print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"Total epochs trained: {len(history.history['loss'])}")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Final learning rate: {float(model.optimizer.learning_rate):.6f}")
print("=" * 70)

In [ ]:
# TRAINING CURVES

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# X-axis: epochs
epochs_range = range(1, len(history.history['accuracy']) + 1)

# Plot training accuracy
ax.plot(epochs_range, history.history['accuracy'],
        color='#2563eb', linewidth=2, label='Training Accuracy')

# Plot validation accuracy
ax.plot(epochs_range, history.history['val_accuracy'],
        color='#eab308', linewidth=2, label='Validation Accuracy')

# Formatting
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.5, 1.0])
ax.set_xlim([1, len(epochs_range)])

plt.tight_layout()

# Save figure
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight', facecolor='white')
print("Figure saved: training_curves.png")

plt.show()

In [ ]:

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Extract values from sklearn's format
tn, fp, fn, tp = cm.ravel()

cm_reordered = np.array([[tp, fn],
                          [fp, tn]])

# Create figure
fig, ax = plt.subplots(figsize=(6, 5))

# Create heatmap
im = ax.imshow(cm_reordered, cmap='Blues')

# Add text annotations
labels = [['TP', 'FN'],
          ['FP', 'TN']]

for i in range(2):
    for j in range(2):
        value = cm_reordered[i, j]
        label = labels[i][j]
        color = 'white' if value > cm_reordered.max() / 2 else 'black'
        ax.text(j, i, f'{value}\n({label})', ha='center', va='center',
                fontsize=14, color=color, fontweight='bold')

# Formatting
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Approved (1)', 'Rejected (0)'], fontsize=11)
ax.set_yticklabels(['Approved (1)', 'Rejected (0)'], fontsize=11)
ax.set_xlabel('Predicted Decision', fontsize=12)
ax.set_ylabel('Actual Outcome', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)

plt.tight_layout()

# Save figure
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight', facecolor='white')
print("Figure saved: confusion_matrix.png")

# Print values for thesis reference
print(f"\nValues: TP={tp}, FN={fn}, FP={fp}, TN={tn}")

plt.show()

In [ ]:
# Model Statistics
print("="*60)
print("MODEL STATISTICS")
print("="*60)

# Total trainable parameters
total_params = model.count_params()
trainable_params = sum([tf.reduce_prod(w.shape).numpy() for w in model.trainable_weights])

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Model architecture summary
print(f"\nArchitecture: {X_train.shape[1]} -> 32 -> 16 -> 1")
print(f"Input features: {X_train.shape[1]}")

# Approval rates
dataset_approval_rate = df['class'].mean()
predicted_approval_rate = y_pred.mean()

print(f"\nDataset approval rate (ground truth): {dataset_approval_rate:.1%}")
print(f"Model predicted approval rate (test set): {predicted_approval_rate:.1%}")
print(f"Test set size: {len(y_pred)} samples")


In [ ]:
np.mean(model.predict(X_train))

# SHAP Explanation Generation

In [ ]:
# Initialize SHAP explainer (DeepSHAP)
background = X_train.values[:100]
explainer = shap.DeepExplainer(model, background)

# Generate SHAP values for test set
shap_values_array = explainer.shap_values(X_test.values)

# Handle SHAP output format
if isinstance(shap_values_array, list):
    shap_values_array = shap_values_array[0]

# Flatten SHAP values if they have extra dimension
if len(shap_values_array.shape) == 3:
    shap_values_array = shap_values_array.squeeze()

# Convert to DataFrame with preserved indices
shap_values = pd.DataFrame(shap_values_array, columns=X_test.columns, index=X_test.index)

print(f"✓ SHAP values generated")
print(f"SHAP values shape: {shap_values.shape}")
print(f"SHAP indices preserved: {shap_values.index[:5].tolist()}")

# Convert expected_value to scalar
expected_val = explainer.expected_value
if isinstance(expected_val, (list, tuple)):
    expected_val = expected_val[0]
if hasattr(expected_val, 'numpy'):
    expected_val = float(expected_val.numpy())
else:
    expected_val = float(np.array(expected_val).flatten()[0])

print(f"Expected value (base rate): {expected_val:.3f}")


# Scenario Selection Algorithm

In [ ]:
def calculate_scenario_quality_score(idx, X_test, y_pred_proba, shap_values, y_pred):
    """
    Calculate quality score for a scenario.
    """
    score = 0

    # 1. Prediction confidence
    confidence = abs(y_pred_proba.loc[idx] - 0.5)
    score += confidence * 100

    # 2. SHAP explanation clarity
    abs_shap = np.abs(shap_values.loc[idx].values)
    top_3_shap = np.sort(abs_shap)[-3:].sum()
    total_shap = abs_shap.sum()
    if total_shap > 0:
        concentration = top_3_shap / total_shap
        score += concentration * 50

    return score


def select_diverse_scenarios(candidate_indices, X_test_original, n_select=3):
    """
    Select diverse scenarios from candidates.
    """
    if len(candidate_indices) <= n_select:
        return candidate_indices

    selected = [candidate_indices[0]]
    remaining = candidate_indices[1:]

    key_features = ['credit_amount', 'duration', 'age', 'purpose', 'credit_history']

    while len(selected) < n_select and len(remaining) > 0:
        best_candidate = None
        best_diversity_score = -1

        for candidate in remaining:
            diversity_score = 0
            for feature in key_features:
                if feature in X_test_original.columns:
                    candidate_val = X_test_original.loc[candidate, feature]
                    for selected_idx in selected:
                        selected_val = X_test_original.loc[selected_idx, feature]
                        if candidate_val != selected_val:
                            diversity_score += 1

            if diversity_score > best_diversity_score:
                best_diversity_score = diversity_score
                best_candidate = candidate

        if best_candidate is not None:
            selected.append(best_candidate)
            remaining.remove(best_candidate)
        else:
            break

    return selected

print("✓ Scenario quality and diversity functions defined")


## Visualization Functions


In [ ]:
def visualize_level1_no_explanation(idx, y_pred, y_pred_proba, save_path=None):
    """
    Level 1: The Amber System.
    """
    decision = "APPROVED" if y_pred.loc[idx] == 1 else "REJECTED"
    color = '#2E7D32' if y_pred.loc[idx] == 1 else '#C62828'

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')

    # Title
    title_text = f"Loan Application {decision}"
    ax.text(0.5, 0.75, title_text,
            ha='center', va='center', fontsize=20, fontweight='bold', color=color)

    # Generic statement
    statement = "This decision was made using AI technology\nbased on historical applicant data."
    ax.text(0.5, 0.35, statement,
            ha='center', va='center', fontsize=12, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)

    plt.show()
    return fig



In [ ]:
def generate_level2_explanation(idx, y_pred, shap_values, X_test, X_test_original):
    """
    level 2: The Opal System. Generate plain English explanation based on top 2 SHAP features.
    """

    decision = "approved" if y_pred.loc[idx] == 1 else "rejected"

    # Get top 2 SHAP features by absolute value
    shap_instance = shap_values.loc[idx].values
    feature_importance = list(zip(X_test.columns.tolist(), shap_instance))
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

    top_feature_1 = feature_importance[0][0]
    top_feature_2 = feature_importance[1][0]
    shap_value_1 = float(feature_importance[0][1])
    shap_value_2 = float(feature_importance[1][1])

    # Get actual values from original data
    row = X_test_original.loc[idx]

    def describe_feature_neutral(feature_name, row):
        """
        Simple, neutral description of feature value.
        """

        value = row[feature_name]

        # For numerical features, include the actual value
        if feature_name == 'credit_amount':
            return f"the loan amount of €{int(value):,}"

        elif feature_name == 'duration':
            return f"the loan duration of {int(value)} months"

        elif feature_name == 'age':
            return f"the applicant's age of {int(value)} years"

        # For categorical features, translate the code
        elif feature_name == 'employment':
            emp_text = VALUE_MAPPINGS.get('employment', {}).get(value, str(value))
            return f"employment duration of {emp_text}"

        elif feature_name == 'savings_status':
            sav_text = VALUE_MAPPINGS.get('savings_status', {}).get(value, str(value))
            return f"savings account status ({sav_text})"

        elif feature_name == 'credit_history':
            return "the applicant's credit history"

        elif feature_name == 'other_payment_plans':
            if value == 'A143':
                return "having no other payment plans"
            else:
                return "having other payment plans"

        elif feature_name == 'existing_credits':
            if int(value) == 1:
                return "having one existing credit"
            else:
                return f"having {int(value)} existing credits"

        elif feature_name == 'installment_rate':
            return f"the installment rate of {int(value)}%"

        elif feature_name == 'purpose':
            purpose_text = VALUE_MAPPINGS.get('purpose', {}).get(value, str(value))
            return f"the loan purpose ({purpose_text})"

        elif feature_name == 'housing':
            housing_text = VALUE_MAPPINGS.get('housing', {}).get(value, str(value))
            if housing_text == 'own':
                return "owning a home"
            elif housing_text == 'rent':
                return "renting"
            else:
                return f"housing status ({housing_text})"

        elif feature_name == 'job':
            job_text = VALUE_MAPPINGS.get('job', {}).get(value, str(value))
            if 'unemployed' in job_text.lower():
                return "employment status (unemployed)"
            else:
                return "the applicant's job status"

        elif feature_name == 'property_magnitude':
            return "the applicant's property ownership"

        elif feature_name == 'personal_status':
            return "the applicant's personal status"

        elif feature_name == 'other_parties':
            if value == 'A101':
                return "having no co-applicant or guarantor"
            else:
                return "having a co-applicant or guarantor"

        elif feature_name == 'residence_since':
            return f"residence duration at current address ({int(value)} years)"

        elif feature_name == 'num_dependents':
            if int(value) == 1:
                return "having 1 dependent"
            else:
                return f"having {int(value)} dependents"

        else:
            # Fallback to readable feature name
            return FEATURE_NAMES.get(feature_name, feature_name).lower()

    # Generate neutral descriptions for top 2 factors
    factor_1_desc = describe_feature_neutral(top_feature_1, row)
    factor_2_desc = describe_feature_neutral(top_feature_2, row)

    # Create neutral explanation
    if decision == "approved":
        explanation = f"The loan application was approved based on {factor_1_desc} and {factor_2_desc}."
    else:
        explanation = f"The loan was rejected based on {factor_1_desc} and {factor_2_desc}."

    return explanation


print("✓ Level 2 explanation generator defined")


In [ ]:
def visualize_level2_simplified_explanation(idx, y_pred, y_pred_proba, shap_values, X_test, X_test_original, save_path=None):
    """
    Level 2: Simplified plain-English SHAP explanation.
    Dynamically generates explanation based on top 2 SHAP features.
    """

    decision = "APPROVED" if y_pred.loc[idx] == 1 else "REJECTED"
    decision_color = '#2E7D32' if y_pred.loc[idx] == 1 else '#C62828'

    # Generate dynamic explanation
    explanation_text = generate_level2_explanation(idx, y_pred, shap_values, X_test, X_test_original)

    # Create visualization
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.axis('off')

    # Title
    ax.text(0.5, 0.85, 'Loan Decision',
            ha='center', va='top', fontsize=16, fontweight='bold')

    # Decision box
    decision_box = plt.Rectangle((0.25, 0.55), 0.5, 0.18,
                                  facecolor=decision_color, alpha=0.15,
                                  edgecolor=decision_color, linewidth=2)
    ax.add_patch(decision_box)

    ax.text(0.5, 0.64, decision,
            ha='center', va='center', fontsize=20, fontweight='bold', color=decision_color)

    # Explanation section
    ax.text(0.5, 0.42, 'Explanation:',
            ha='center', va='top', fontsize=13, fontweight='bold')

    # Wrap explanation text for better readability
    import textwrap
    wrapped_text = textwrap.fill(explanation_text, width=70)

    ax.text(0.5, 0.35, wrapped_text,
            ha='center', va='top', fontsize=11, style='italic',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', alpha=0.3))

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  → Level 2 saved: {save_path}")

    plt.show()
    plt.close()

print("✓ Level 2 visualization function updated (dynamic explanations)")



In [ ]:
def visualize_level3_detailed_shap(idx, y_pred, y_pred_proba, shap_values, feature_names,
                                   X_test_scaled, explainer, save_path=None):
    """
    Level 3: The Quartz System. Detailed technical SHAP waterfall plot.
    """
    decision = "APPROVED" if y_pred.loc[idx] == 1 else "REJECTED"
    color = '#2E7D32' if y_pred.loc[idx] == 1 else '#C62828'

    # Convert base_values to float
    base_val = explainer.expected_value
    if isinstance(base_val, (list, tuple)):
        base_val = base_val[0]
    if hasattr(base_val, 'numpy'):
        base_val = float(base_val.numpy())
    else:
        base_val = float(np.array(base_val).flatten()[0])

    # Ensure both data and SHAP values are 1D arrays
    data_1d = X_test_scaled.loc[idx].values.flatten() if hasattr(X_test_scaled.loc[idx].values, 'flatten') else X_test_scaled.loc[idx].values
    shap_vals_1d = shap_values.loc[idx].values

    # Create explanation object for waterfall
    explanation = shap.Explanation(
        values=shap_vals_1d,
        base_values=base_val,
        data=data_1d,
        feature_names=feature_names
    )

    # Create figure
    fig = plt.figure(figsize=(12, 10))

    # Add title and confidence at the top
    plt.suptitle(f"Loan Application {decision}",
                 fontsize=16, fontweight='bold', color=color, y=0.98)

    # Create waterfall plot
    shap.plots.waterfall(explanation, max_display=15, show=False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)

    plt.show()
    return fig




## Loan Applicant Profile Generator


In [ ]:
def generate_applicant_profile(idx, X_test_original):
    """Generate natural language applicant profile."""

    row = X_test_original.loc[idx]

    # Basic info
    age = row['age']
    loan_amount = row['credit_amount']
    duration = row['duration']

    # Checking status mapping
    checking_map = VALUE_MAPPINGS['checking_status']
    checking_status = checking_map.get(row['checking_status'], row['checking_status'])

    # Purpose mapping
    purpose_map = VALUE_MAPPINGS['purpose']
    purpose = purpose_map.get(row['purpose'], row['purpose'])

    # Employment mapping
    employment_map = VALUE_MAPPINGS['employment']
    employment = employment_map.get(row['employment'], row['employment'])

    # Savings mapping
    savings_map = VALUE_MAPPINGS['savings_status']
    savings = savings_map.get(row['savings_status'], row['savings_status'])

    # Credit history
    credit_hist_map = VALUE_MAPPINGS['credit_history']
    credit_history = credit_hist_map.get(row['credit_history'], row['credit_history'])

    # Build profile text
    profile = f"""
**Applicant Profile:**

- **Age:** {age} years old
- **Loan Amount Requested:** EUR{loan_amount:,}
- **Loan Duration:** {duration} months
- **Purpose:** {purpose.capitalize()}
- **Employment Duration:** {employment.capitalize()}
- **Checking Account Status:** {checking_status.capitalize()}
- **Savings:** {savings.capitalize()}
- **Credit History:** {credit_history.capitalize()}
    """.strip()

    return profile

print("Applicant profile generator defined")

# Create output directory
output_dir = "loan_approval_scenarios_v6"
os.makedirs(output_dir, exist_ok=True)

# Select 6 optimal scenarios (3 approved, 3 rejected)
print("\n" + "="*60)
print("SELECTING OPTIMAL SCENARIOS")
print("="*60)

# Calculate quality scores for all test samples
approved_scores = []
rejected_scores = []

for idx in X_test.index:
    score = calculate_scenario_quality_score(idx, X_test, y_pred_proba, shap_values, y_pred)

    if y_pred.loc[idx] == 1:  # Approved
        approved_scores.append((idx, score))
    else:  # Rejected
        rejected_scores.append((idx, score))

# Sort by score and select top 3 from each category
approved_scores.sort(key=lambda x: x[1], reverse=True)
rejected_scores.sort(key=lambda x: x[1], reverse=True)

top_approved = [idx for idx, score in approved_scores[:3]]
top_rejected = [idx for idx, score in rejected_scores[:3]]

# Combine all scenarios
all_scenarios = top_approved + top_rejected

print(f"\nSelected 3 approved scenarios: {top_approved}")
print(f"Selected 3 rejected scenarios: {top_rejected}")

# Generate visualizations for all scenarios
print("\n" + "="*60)
print("GENERATING SCENARIO VISUALIZATIONS")
print("="*60)

for scenario_num, idx in enumerate(all_scenarios, 1):
    print(f"\n--- Scenario {scenario_num} (Test Index: {idx}) ---")

    # Generate applicant profile
    profile = generate_applicant_profile(idx, X_test_original)
    print(profile)

    # Export profile to text file
    profile_path = f"{output_dir}/scenario_{scenario_num}_applicant_profile.txt"
    with open(profile_path, 'w') as f:
        f.write(profile)

    decision = "Approved" if y_pred.loc[idx] == 1 else "Rejected"
    print(f"\n**Decision:** {decision}")

    # Level 1: No explanation
    level1_path = f"{output_dir}/scenario_{scenario_num}_level1_no_explanation.png"
    visualize_level1_no_explanation(idx, y_pred, y_pred_proba, save_path=level1_path)

    # Level 2: Simplified explanation
    level2_path = f"{output_dir}/scenario_{scenario_num}_level2_simplified.png"
    visualize_level2_simplified_explanation(idx, y_pred, y_pred_proba, shap_values, X_test, X_test_original, save_path=level2_path)

    # Level 3: Detailed SHAP
    level3_path = f"{output_dir}/scenario_{scenario_num}_level3_detailed_shap.png"
    visualize_level3_detailed_shap(idx, y_pred, y_pred_proba, shap_values,
                                   feature_names=X_test.columns.tolist(),
                                   X_test_scaled=X_test,
                                   explainer=explainer,
                                   save_path=level3_path)

    print()

print("="*60)
print("ALL SCENARIOS GENERATED SUCCESSFULLY")
print("="*60)
print(f"\nOutput directory: {output_dir}/")
print("\nFiles generated per scenario:")
print("  - scenario_X_level1_no_explanation.png")
print("  - scenario_X_level2_simplified.png")
print("  - scenario_X_level3_detailed_shap.png")



## Generate All Scenarios


In [ ]:
# Create output directory
output_dir = "loan_approval_scenarios_v5"
os.makedirs(output_dir, exist_ok=True)

# Select 6 optimal scenarios (3 approved, 3 rejected)
print("\n" + "="*60)
print("SELECTING OPTIMAL SCENARIOS")
print("="*60)

# Calculate quality scores for all test samples
approved_scores = []
rejected_scores = []

for idx in X_test.index:
    score = calculate_scenario_quality_score(idx, X_test, y_pred_proba, shap_values, y_pred)

    if y_pred.loc[idx] == 1:  # Approved
        approved_scores.append((idx, score))
    else:  # Rejected
        rejected_scores.append((idx, score))

# Sort by score and select top 3 from each category
approved_scores.sort(key=lambda x: x[1], reverse=True)
rejected_scores.sort(key=lambda x: x[1], reverse=True)

top_approved = [idx for idx, score in approved_scores[:3]]
top_rejected = [idx for idx, score in rejected_scores[:3]]

# Combine all scenarios
all_scenarios = top_approved + top_rejected

print(f"\n✓ Selected 3 approved scenarios: {top_approved}")
print(f"✓ Selected 3 rejected scenarios: {top_rejected}")

# Generate visualizations for all scenarios
print("\n" + "="*60)
print("GENERATING SCENARIO VISUALIZATIONS")
print("="*60)

for scenario_num, idx in enumerate(all_scenarios, 1):
    print(f"\n--- Scenario {scenario_num} (Test Index: {idx}) ---")

    # Generate applicant profile
    profile = generate_applicant_profile(idx, X_test_original)
    print(profile)

    # Export profile to text file
    profile_path = f"{output_dir}/scenario_{scenario_num}_applicant_profile.txt"
    with open(profile_path, 'w') as f:
        f.write(profile)

    decision = "Approved" if y_pred.loc[idx] == 1 else "Rejected"
    print(f"\n**Decision:** {decision}")

    # Level 1: The Amber System
    level1_path = f"{output_dir}/scenario_{scenario_num}_level1_no_explanation.png"
    visualize_level1_no_explanation(idx, y_pred, y_pred_proba, save_path=level1_path)

    # Level 2: the Opal Systme
    level2_path = f"{output_dir}/scenario_{scenario_num}_level2_simplified.png"
    visualize_level2_simplified_explanation(idx, y_pred, y_pred_proba, shap_values, X_test, X_test_original, save_path=level2_path)

    # Level 3: the Quartz System
    level3_path = f"{output_dir}/scenario_{scenario_num}_level3_detailed_shap.png"
    visualize_level3_detailed_shap(idx, y_pred, y_pred_proba, shap_values,
                                   feature_names=X_test.columns.tolist(),
                                   X_test_scaled=X_test,
                                   explainer=explainer,
                                   save_path=level3_path)

    print()

print("="*60)
print("✓ ALL SCENARIOS GENERATED SUCCESSFULLY")
print("="*60)
print(f"\nOutput directory: {output_dir}/")
print("\nFiles generated per scenario:")
print("  - scenario_X_level1_no_explanation.png")
print("  - scenario_X_level2_simplified.png")
print("  - scenario_X_level3_detailed_shap.png")



## Summary

In [ ]:
# Generate summary report
print("\n" + "="*70)
print("Scenario Summary")
print("="*70)

for scenario_num, idx in enumerate(all_scenarios, 1):
    decision = "APPROVED" if y_pred.loc[idx] == 1 else "REJECTED"
    confidence = y_pred_proba.loc[idx] if y_pred.loc[idx] == 1 else (1 - y_pred_proba.loc[idx])

    print(f"\nScenario {scenario_num}: {decision} (Confidence: {confidence:.1%})")

    # Show top 3 features
    shap_instance = shap_values.loc[idx].values
    feature_importance = list(zip(X_test.columns.tolist(), shap_instance))
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

    print("  Top influencing factors:")
    for feat, shap_val in feature_importance[:3]:
        readable_name = FEATURE_NAMES.get(feat, feat)
        shap_val_float = float(shap_val) if hasattr(shap_val, '__float__') else shap_val
        print(f"    - {readable_name}: {shap_val_float:+.3f}")




# Create metadata DataFrame for all scenarios
metadata_list = []

for scenario_num, idx in enumerate(all_scenarios, 1):
    row = X_test_original.loc[idx]
    decision = "Approved" if y_pred.loc[idx] == 1 else "Rejected"
    confidence = y_pred_proba.loc[idx] if y_pred.loc[idx] == 1 else (1 - y_pred_proba.loc[idx])

    # Get top 3 SHAP features
    shap_instance = shap_values.loc[idx].values
    feature_importance = list(zip(X_test.columns.tolist(), shap_instance))
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

    metadata = {
        'Scenario_Number': scenario_num,
        'Test_Index': idx,
        'Decision': decision,
        'Confidence': f"{confidence:.1%}",
        'Prediction_Probability': f"{y_pred_proba.loc[idx]:.3f}",
        'Age': row['age'],
        'Loan_Amount': row['credit_amount'],
        'Duration_Months': row['duration'],
        'Purpose': row['purpose'],
        'Employment': row['employment'],
        'Savings': row['savings_status'],
        'Credit_History': row['credit_history'],
        'Top_Feature_1': FEATURE_NAMES.get(feature_importance[0][0], feature_importance[0][0]),
        'Top_Feature_1_SHAP': f"{float(feature_importance[0][1]):.4f}",
        'Top_Feature_2': FEATURE_NAMES.get(feature_importance[1][0], feature_importance[1][0]),
        'Top_Feature_2_SHAP': f"{float(feature_importance[1][1]):.4f}",
        'Top_Feature_3': FEATURE_NAMES.get(feature_importance[2][0], feature_importance[2][0]),
        'Top_Feature_3_SHAP': f"{float(feature_importance[2][1]):.4f}",
    }

    metadata_list.append(metadata)

# Create DataFrame and save
metadata_df = pd.DataFrame(metadata_list)
metadata_path = f"{output_dir}/scenario_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)

print("✓ Scenario metadata exported")
print(f"Location: {metadata_path}")
print("\nMetadata preview:")
metadata_df


In [ ]:
# Create metadata DataFrame for all scenarios
metadata_list = []

for scenario_num, idx in enumerate(all_scenarios, 1):
    row = X_test_original.loc[idx]
    decision = "Approved" if y_pred.loc[idx] == 1 else "Rejected"
    confidence = y_pred_proba.loc[idx] if y_pred.loc[idx] == 1 else (1 - y_pred_proba.loc[idx])

    # Get top 3 SHAP features
    shap_instance = shap_values.loc[idx].values
    feature_importance = list(zip(X_test.columns.tolist(), shap_instance))
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

    metadata = {
        'Scenario_Number': scenario_num,
        'Test_Index': idx,
        'Decision': decision,
        'Confidence': f"{confidence:.1%}",
        'Prediction_Probability': f"{y_pred_proba.loc[idx]:.3f}",
        'Age': row['age'],
        'Loan_Amount': row['credit_amount'],
        'Duration_Months': row['duration'],
        'Purpose': row['purpose'],
        'Employment': row['employment'],
        'Savings': row['savings_status'],
        'Credit_History': row['credit_history'],
        'Top_Feature_1': FEATURE_NAMES.get(feature_importance[0][0], feature_importance[0][0]),
        'Top_Feature_1_SHAP': f"{float(feature_importance[0][1]):.4f}",
        'Top_Feature_2': FEATURE_NAMES.get(feature_importance[1][0], feature_importance[1][0]),
        'Top_Feature_2_SHAP': f"{float(feature_importance[1][1]):.4f}",
        'Top_Feature_3': FEATURE_NAMES.get(feature_importance[2][0], feature_importance[2][0]),
        'Top_Feature_3_SHAP': f"{float(feature_importance[2][1]):.4f}",
    }

    metadata_list.append(metadata)

# Create DataFrame and save
metadata_df = pd.DataFrame(metadata_list)
metadata_path = f"{output_dir}/scenario_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)

print("✓ Scenario metadata exported")
print(f"Location: {metadata_path}")
print("\nMetadata preview:")
metadata_df
